# AMADS Coding Notebooks

### Chord Equivalence over longer sequences (ngrams)

The [`chord_bigram.py`](https://github.com/music-computing/amads/blob/main/amads/harmony/chord_bigram.py)
module sets up Murphy's Ø / I / R / K taxonomy for equivalence among chord *pairs*.

(Note that there is a notebook at specifically for this at
[`notebooks/chord_bigram.ipynb`](https://github.com/music-computing/amads/blob/main/notebooks/chord_bigram.ipynb)).

The [`chord_ngram.py`](https://github.com/music-computing/amads/blob/main/amads/harmony/chord_ngram.py)
module generalises this to **three or more** chords.
This requires a separate axis for chord successions (Ø, R, C, D, P for *order*),
which combine independently "", I, K, IK for *content*,
to make for $5*4=20$ in total.

This notebook illustrates the above equivalence classes, mostly with heatmaps.
Throughout:
- the "task" is exploration only,
- the aim is to demonstrate how equivalence relation partition the space into classes,
- we use heat maps as a useful "at a glance" summary of those partitions.
- cells that share a value (visualised with colour) are equivalent under that regime.
- moving to coarser regimes, cells merge and the picture gets "blockier".

---

**By (author/s):** Mark Gotham

**For:** Attached to the
[AMADS code library](https://github.com/music-computing/amads/) and
["Keeping Score" book](https://doi.org/10.5281/zenodo.14938027),
but open to all.

**Licence:** MIT.

**Colour key (here green only):**
- <font color='green'> Green is for a block of information.
- <font color='purple'> Purple is for an exercise.
- <font color='crimson'> Crimson is for the solution to the exercise above it.

---

# Set up

In [ ]:
# Plotting import and parameters

import matplotlib.pyplot as plt

# Disable grid lines globally for all plots (highly recommended for this notebook):
plt.rcParams['axes.grid'] = False

# Other options include:
# plt.rcParams["figure.dpi"] = 110


In [ ]:
# Other
import itertools
import numpy as np

from amads.core.chord import Chord
from amads.harmony.chord_ngram import ChordNgram, _CONTENT_CODES, _ORDER_CODES

NOTE_NAMES = ["C","C#","D","D#","E","F","F#","G","G#","A","A#","B"]

## <font color='green'> 1. Bigrams: transposition (K) and retrograde (R) as banding & symmetry


For every pair of major-triad roots $(i, j) \in \{0,\dots,11\}^2$,
we build a `ChordNgram([Chord(i), Chord(j)], regime)`
and colour cell $(i,j)$ by an integer id for its canonical form.
Two cells share a colour iff the two bigrams are equivalent under that regime.

- **Ø** (exact):
    - all 144 cells have their own class (no structure).
- **K** (transposition only):
    - canonical form depends only on $j - i \bmod 12$,
    - cells on the same *diagonal* share a colour
    - **diagonal banding**.
- **R** (retrograde only):
    - canonical form is symmetric in $(i,j)$,
    - cell $(i,j)$ always matches cell $(j,i)$
    - **mirror symmetry about the leading diagonal**.
- **KR** (both):
    - banding *and* symmetry combine,
    - this collapses the 144 cells into just 7 interval classes.

In [ ]:
def bigram_class_grid(regime, quality="major"):
    """12x12 grid of canonical-class ids for root pairs (i, j)"""
    grid = np.zeros((12, 12), dtype=int)
    seen = {}
    for i in range(12):
        for j in range(12):
            ng = ChordNgram([Chord(i, quality), Chord(j, quality)], regime)
            key = ng.canonical
            if key not in seen:
                seen[key] = len(seen)
            grid[i, j] = seen[key]
    return grid, len(seen)


fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, regime in zip(axes, ["\u00d8", "K", "R", "KR"]):
    grid, n_classes = bigram_class_grid(regime)
    ax.imshow(grid, cmap="nipy_spectral")
    ax.set_title(f"{regime}  ({n_classes} classes)")
    ax.set_xlabel("chord$_2$ root")
    ax.set_ylabel("chord$_1$ root")
    ax.set_xticks(range(12)); ax.set_xticklabels(NOTE_NAMES, rotation=90, fontsize=7)
    ax.set_yticks(range(12)); ax.set_yticklabels(NOTE_NAMES, fontsize=7)
fig.suptitle("Bigram canonical class, all-major root pairs (i, j)", y=1.05)
fig.tight_layout()
plt.show()


The class counts (144 -> 12 -> 78 -> 7) match the group sizes exactly:
- K has 12 directed intervals;
- R merges every off-diagonal cell with its mirror ($ (144-12)/2 + 12 = 78 $);
- KR has $\lceil 12/2 \rceil + 1 = 7$ interval *classes* (0 through 6).


## <font color='green'> 2. Inversion (I): a hidden symmetry that only shows up across quality


Here (after Murphy) I flips every chord's quality (major->minor and vice versa).

In the above we restrict ourselves to all-major chords so "I" doesn't feature.

In fact, the all-major scenario above in its entirety is I-equivalent to the all-minor.
This forms part of the total I-equivalence across the major and minor scenario which halves the number of distinct cases.

Using the full 24-chord alphabet (12 roots $\times$ 2 qualities)
on each axis makes the 2-for-1 merging visible directly.

In [ ]:
chords24 = [Chord(pc, q) for pc in range(12) for q in ("major", "minor")]
labels24 = [f"{NOTE_NAMES[pc]}{'M' if q=='major' else 'm'}" for pc in range(12) for q in ("major", "minor")]

def bigram_class_grid_24(regime):
    grid = np.zeros((24, 24), dtype=int)
    seen = {}
    for i, a in enumerate(chords24):
        for j, b in enumerate(chords24):
            key = ChordNgram([a, b], regime).canonical
            if key not in seen:
                seen[key] = len(seen)
            grid[i, j] = seen[key]
    return grid, len(seen)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, regime in zip(axes, ["\u00d8", "I"]):
    grid, n_classes = bigram_class_grid_24(regime)
    ax.imshow(grid, cmap="nipy_spectral")
    ax.set_title(f"{regime}  ({n_classes} classes)")
    ax.set_xticks(range(24)); ax.set_xticklabels(labels24, rotation=90, fontsize=6)
    ax.set_yticks(range(24)); ax.set_yticklabels(labels24, fontsize=6)
fig.suptitle("Full 24-chord alphabet: I exactly halves the class count (576 \u2192 288)", y=1.03)
fig.tight_layout()
plt.show()


## <font color='green'> 3. Order equivalence for longer successions: Ø, R, C, D, P


Let's take a 4-chord succession of distinct chords.

Naturally,we'll use a very common loop (a-F-C-G)
though it actually doesn't matter which chords for this specific demo.

We'll look at all $4! = 24$ reorderings of this 4.

For each order-equivalence regime,
build the $24\times24$ pairwise-equality matrix (white = equivalent).
As the regime coarsens (Ø $\to$ R $\to$ C $\to$ D $\to$ P),
the matrix should visibly fill in with ever-larger equal-sized blocks.


In [ ]:
a_F_C_G = [
    Chord(9, "minor"),
    Chord(5, "major"),
    Chord(0, "major"),
    Chord(7, "major")]
perms = list(itertools.permutations(a_F_C_G))

fig, axes = plt.subplots(1, 5, figsize=(19, 4))
for ax, regime in zip(axes, ["\u00d8", "R", "C", "D", "P"]):
    seqs = [ChordNgram(list(p), regime) for p in perms]
    eq = np.array([[1 if a == b else 0 for b in seqs] for a in seqs])
    n_classes = len({s.canonical for s in seqs})
    ax.imshow(eq, cmap="Greys")
    ax.set_title(f"{regime}  ({n_classes} classes)")
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Pairwise equality among all 24 orderings of a 4-chord succession", y=1.05)
fig.tight_layout()
plt.show()


Class counts: **24 -> 12 -> 6 -> 3 -> 1**,
i.e. exactly $24/|G|$ for group sizes $|G| = 1, 2, 4, 8, 24$.

R, C, D, P are subgroups of increasing size of the symmetric group $S_4$,
nested as the theory predicts ($R, C \subset D \subset P$).


## <font color='green'> 4. The full content $\times$ order lattice, in one heat map


`chord_ngram.py` combines the
_content_ axis ("", I, K, IK) with the
_order_ axis (Ø, R, C, D, P)
into $4*5=20$ regimes.

First fix a seed succession (C, E, G major) and
generate its full orbit under the *largest* combined group
(IKP: every transposition, the global quality flip, and every reordering)
to yield 144 distinct raw successions.

Then, for **each of the 20 regimes**,
count how many distinct classes that same 144-member orbit collapses into.

This is the coarsening lattice from `ChordNgram.coarsen()` made visible:
counts should decrease monotonically as you move
- right (Ø -> R -> C -> D -> P) and
- down ("" -> I/K -> IK), and therefore,
- certainly both (down-right).

In [ ]:
orbit = set()
for k in range(12):
    for flip in (False, True):
        for perm in itertools.permutations(range(3)):
            chs = []
            for idx in perm:
                # Note: Re-use the a_F_C_G test case
                pc, q = a_F_C_G[idx].root.pitch_class, a_F_C_G[idx].quality
                pc2 = (pc + k) % 12
                q2 = ("minor" if q == "major" else "major") if flip else q
                chs.append((pc2, q2))
            orbit.add(tuple(chs))

order_suffix = {"\u00d8": "", "R": "R", "C": "C", "D": "D", "P": "P"}
grid = np.zeros((4, 5), dtype=int)
for ci, content in enumerate(_CONTENT_CODES):
    for oi, order in enumerate(_ORDER_CODES):
        label = content + order_suffix[order]
        label = label if label else "\u00d8"
        classes = {ChordNgram([Chord(pc, q) for pc, q in seq], label).canonical for seq in orbit}
        grid[ci, oi] = len(classes)

fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(grid, cmap="viridis")
ax.set_xticks(range(5)); ax.set_xticklabels(_ORDER_CODES)
ax.set_yticks(range(4)); ax.set_yticklabels(['""'] + list(_CONTENT_CODES[1:]))
ax.set_xlabel("order axis"); ax.set_ylabel("content axis")
for ci in range(4):
    for oi in range(5):
        ax.text(oi, ci, grid[ci, oi], ha="center", va="center",
                 color="white" if grid[ci, oi] < grid.max() * 0.6 else "black")
ax.set_title(f"Distinct classes within the {len(orbit)}-member orbit a 4-chord progression")
fig.colorbar(im, ax=ax, label="# distinct classes", shrink=0.8)
fig.tight_layout()
plt.show()

## <font color='green'> 5. Periodicity and palindromes shrink the order group further

`ChordNgram.order_group_size` counts *distinct* orderings reachable under a regime
(automatically accounting for repeated chords).

Here are some progression shapes with internal regularity that reduce the distinct options further:
- **ABAB** has an internal pattern of period 2 and therefore has only 2 distinct rotations, not 4.
- **ABCBA** is its own retrograde (a palindrome), so R pairs it with itself, not a different succession.
- **AAAA** is ... well ... all of the above.

Let's see them in heatmap action.

In [ ]:
C, E, G, A = Chord(0, "major"), Chord(4, "major"), Chord(7, "major"), Chord(9, "major")
shapes = {
    "ABCD (all distinct)": [C, E, G, A],
    "ABAB (period 2)": [C, E, C, E],
    "AABB": [C, C, E, E],
    "ABCA (one repeat)": [C, E, G, C],
    "AAAA (all same)": [C, C, C, C],
    "ABCBA (palindrome, n=5)": [C, E, G, E, C],
}
regimes = ["\u00d8", "R", "C", "D", "P"]
grid = np.array([[ChordNgram(chs, r).order_group_size for r in regimes] for chs in shapes.values()])

fig, ax = plt.subplots(figsize=(7, 4.5))
im = ax.imshow(grid, cmap="magma")  # Magma is similar to viridis, but arguably highlights extreme values "better".
ax.set_xticks(range(len(regimes))); ax.set_xticklabels(regimes)
ax.set_yticks(range(len(shapes))); ax.set_yticklabels(shapes.keys())
for r in range(grid.shape[0]):
    for c in range(grid.shape[1]):
        ax.text(c, r, grid[r, c], ha="center", va="center",
                 color="white" if grid[r, c] < grid.max() * 0.6 else "black")
ax.set_title("order_group_size: distinct orderings by shape × regime")
fig.colorbar(im, ax=ax, label="distinct orderings", shrink=0.8)
fig.tight_layout()
plt.show()

Ends

---